In [1]:
# Python 3.11 (Google Colab)
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 101.7 MB/s eta 0:00:00


In [2]:
# Python 3.11

from google.colab import files

uploaded = files.upload()

audio_file = next(iter(uploaded))
print(f"Uploaded: {audio_file}")

Saving audio1_fz0uSqx0.mp3 to audio1_fz0uSqx0.mp3
Uploaded: audio1_fz0uSqx0.mp3


In [3]:
# Python 3.11

import re
from pathlib import Path
from faster_whisper import WhisperModel


MODEL_SIZE = "medium"  # tiny, base, small, medium, large-v3

print("Loading model...")
model = WhisperModel(
    MODEL_SIZE,
    device="cuda",
    compute_type="float16"
)

print("Transcribing...")

segments, info = model.transcribe(
    audio_file,
    beam_size=5,
    vad_filter=True,
    word_timestamps=False
)

segments = list(segments)

print(f"Language: {info.language}")
print(f"Probability: {info.language_probability:.2f}")


def format_timestamp(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"[{minutes:02d}:{seconds:02d}]"


output_lines = []

for seg in segments:

    text = seg.text.strip()

    if not text:
        continue

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        continue

    seg_duration = max(0.1, seg.end - seg.start)
    step = seg_duration / len(sentences)

    for i, sentence in enumerate(sentences):

        sentence_time = seg.start + (i * step)

        output_lines.append(
            f"{format_timestamp(sentence_time)} {sentence}"
        )


transcript_text = "\n".join(output_lines)

print("\n===== PREVIEW =====\n")
print(transcript_text[:3000])

Loading model...
Transcribing...
Language: en
Probability: 1.00

===== PREVIEW =====

[00:00] You wake up, your eyes are open, but your body will not move.
[00:04] Somewhere in the dark corner of the room, you feel it.
[00:07] A presence, heavy, watching, sitting on your chest.
[00:12] You try to scream, nothing comes out.
[00:15] Your heart is pounding so hard, you can feel it in your throat.
[00:19] And then, just as suddenly as it began, it's over.
[00:23] You can move again, you're alone in your room.
[00:26] There was never anyone there.
[00:28] But here's the strange part.
[00:30] This exact experience has been described almost word for word
[00:34] by people in nearly every culture on earth for thousands of years.
[00:39] Long before psychology had a name for it.
[00:42] Long before anyone understood what was happening inside a sleeping brain.
[00:46] They just knew something visited them in the night.
[00:49] In medieval Europe, they called it the old hag.
[00:53] A withered wo

In [4]:
# Python 3.11

from google.colab import files
from pathlib import Path

output_file = (
    Path(audio_file).stem +
    "_timestamped_transcript.txt"
)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(transcript_text)

print(f"Saved: {output_file}")

files.download(output_file)

Saved: audio1_fz0uSqx0_timestamped_transcript.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>